# Task 3 -- Domain Generalization on PACS

**Before running:** Settings (right sidebar) -> Accelerator: **GPU T4 x2**, Internet: **On**.

**Inputs you need attached** (Add Input, right sidebar):
1. `task3_pa1.zip` -- this task's code, uploaded as a Kaggle dataset (same way as `task1_pa1.zip` / `task2_pa1.zip`).
2. Your finished **Task 2 outputs**: `task2_checkpoints.zip` AND `task2_results.zip` (the two zips the Task 2 notebook produced in its last cell), uploaded together as ONE new Kaggle dataset. Task 3 needs Task 2's `source_only.pt` checkpoint and its saved train/val split -- it reuses both unchanged rather than recomputing them.
3. A PACS dataset (same one you attached for Task 2).

Every path is auto-discovered with `glob` -- you do not need to know or edit any `/kaggle/input/...` path yourself.

In [ ]:
!nvidia-smi
!curl -sI https://www.google.com | head -1

In [ ]:
# --- Locate this task's code (os.walk-based, robust to any /kaggle/input
#     mount layout, including the nested datasets/<owner>/<slug>/ structure
#     Kaggle sometimes uses -- glob's recursive "**" was observed to miss
#     files under that layout even though os.walk finds them) ---
import os, zipfile, shutil

def find_file(root, filename):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        if filename in filenames:
            hits.append(os.path.join(dirpath, filename))
    return hits

hits = find_file("/kaggle/input", "evaluate_sketch.py")
if hits:
    src_root_dir = os.path.dirname(os.path.dirname(hits[0]))
    print("Found already-extracted code at:", src_root_dir)
    os.makedirs("/kaggle/working/pa3", exist_ok=True)
    shutil.copytree(src_root_dir, "/kaggle/working/pa3", dirs_exist_ok=True)
else:
    zip_candidates = []
    for dirpath, dirnames, filenames in os.walk("/kaggle/input", followlinks=True):
        for fn in filenames:
            if fn.lower().endswith(".zip") and ("task3" in fn.lower() or "pa3" in fn.lower()):
                zip_candidates.append(os.path.join(dirpath, fn))
    assert zip_candidates, "Could not find task3/ (extracted) or task3_pa1.zip under /kaggle/input."
    code_zip = zip_candidates[0]
    print("Using code zip:", code_zip)
    os.makedirs("/kaggle/working/pa3", exist_ok=True)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall("/kaggle/working/pa3")

hits2 = find_file("/kaggle/working/pa3", "evaluate_sketch.py")
assert hits2, "task3/evaluate_sketch.py not found after copy/extract -- check input contents."
task3_dir = os.path.dirname(hits2[0])
root_dir = os.path.dirname(task3_dir)
print("task3/ ready at:", task3_dir)
assert os.path.isdir(os.path.join(root_dir, "shared")), "shared/ not found next to task3/."

%cd {task3_dir}
print("Working directory:", os.getcwd())


In [ ]:
# --- Locate Task 2's source_only checkpoint + saved split (os.walk-based,
#     same rationale as cell 2), and place them where task3/'s relative
#     paths (../task2/checkpoints, ../task2/results) expect ---
import os, shutil, zipfile

def find_file(root, filename):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        if filename in filenames:
            hits.append(os.path.join(dirpath, filename))
    return hits

ckpt_hits = find_file("/kaggle/input", "source_only.pt")
split_hits = find_file("/kaggle/input", "pacs_sketch_seed6304_split.json")

# also check inside any zip file under /kaggle/input, in case the Task 2
# outputs were attached as task2_checkpoints.zip / task2_results.zip rather
# than already-extracted files
if not ckpt_hits or not split_hits:
    for dirpath, dirnames, filenames in os.walk("/kaggle/input", followlinks=True):
        for fn in filenames:
            if fn.lower().endswith(".zip"):
                z = os.path.join(dirpath, fn)
                try:
                    with zipfile.ZipFile(z) as zf:
                        names = zf.namelist()
                        if any(n.endswith("source_only.pt") for n in names) or any(n.endswith("pacs_sketch_seed6304_split.json") for n in names):
                            out = f"/kaggle/working/_task2_extract_{fn}"
                            zf.extractall(out)
                            ckpt_hits += find_file(out, "source_only.pt")
                            split_hits += find_file(out, "pacs_sketch_seed6304_split.json")
                except zipfile.BadZipFile:
                    pass

assert ckpt_hits, "Could not find source_only.pt anywhere under /kaggle/input. Attach your Task 2 checkpoints."
assert split_hits, "Could not find pacs_sketch_seed6304_split.json anywhere under /kaggle/input. Attach your Task 2 results."

task2_ckpt_dir = os.path.join(root_dir, "task2", "checkpoints")
task2_results_dir = os.path.join(root_dir, "task2", "results")
os.makedirs(task2_ckpt_dir, exist_ok=True)
os.makedirs(task2_results_dir, exist_ok=True)
shutil.copy(ckpt_hits[0], os.path.join(task2_ckpt_dir, "source_only.pt"))
shutil.copy(split_hits[0], os.path.join(task2_results_dir, "pacs_sketch_seed6304_split.json"))
print("Copied Task 2 checkpoint from:", ckpt_hits[0])
print("Copied Task 2 split from:", split_hits[0])

In [ ]:
# --- Locate PACS ---
import glob, os
from collections import defaultdict

candidates = defaultdict(int)
for name in ["photo", "art_painting", "cartoon", "sketch"]:
    seen_for_this_name = set()
    for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True):
        seen_for_this_name.add(os.path.dirname(p))
    for p in glob.glob(f"/kaggle/input/**/{name.replace('_', ' ')}", recursive=True):
        seen_for_this_name.add(os.path.dirname(p))
    for root in seen_for_this_name:
        candidates[root] += 1

print("All candidate PACS roots found (how many of the 4 domains matched):")
for root, count in candidates.items():
    print(f"  {count}/4  {root}")

# a valid root needs all 4 domains, and must not be a derived/transformed
# variant (e.g. "dct" = DCT/frequency-transformed images, not real photos --
# some public PACS mirrors bundle both alongside each other)
valid = [r for r, c in candidates.items() if c == 4 and "dct" not in r.lower()]
assert valid, f"No clean 4/4 PACS root found. All candidates: {dict(candidates)}"
if len(valid) > 1:
    print("WARNING: multiple valid candidates, picking the first -- verify this is right:", valid)
pacs_root = valid[0]
print("\nSelected PACS root:", pacs_root)


In [ ]:
# --- Point config.yaml at the discovered PACS root ---
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["dataset"]["root"] = pacs_root
with open("configs/config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("dataset.root set to:", cfg["dataset"]["root"])

In [ ]:
# --- Smoke test (seconds, no real data or Task 2 checkpoint needed) ---
!python tests/test_pipeline.py

If the smoke test above shows any `[FAIL]` lines, stop here before training -- everything below burns real GPU time.

In [ ]:
# ERM: loads Task 2's checkpoint, does not train
!python train.py --method erm

In [ ]:
!python train.py --method dan_dg

In [ ]:
!python train.py --method sam

### Controlled design study: DAN-DG's lambda_dg sweep
`lambda_dg=1.0` is already covered by the main DAN-DG run above, so only the two extra points need training. (If you switched `controlled_study.method` to `sam` in the config instead, run the SAM `--rho` sweep here -- see task3/README.md.)

In [ ]:
!python train.py --method dan_dg --lambda_dg 0.1  --tag _lam0.1

In [ ]:
!python train.py --method dan_dg --lambda_dg 10.0 --tag _lam10.0

In [ ]:
# THE ONLY CELL THAT LOADS SKETCH -- run only after everything above is finished
!python evaluate_sketch.py

In [ ]:
# --- Zip everything you need to download ---
import shutil
shutil.make_archive("/kaggle/working/task3_results", "zip", ".", "results")
shutil.make_archive("/kaggle/working/task3_checkpoints", "zip", ".", "checkpoints")
print("Download task3_results.zip and task3_checkpoints.zip from the Output pane on the right.")